In [23]:

import os
# You can create model using this method also,
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

from langchain.chat_models import init_chat_model

In [24]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
model2 = ChatOpenAI(
    model_name="o3-mini",  # Use latest model version                       # Total response length limit
    reasoning_effort='high' # high, medium, low
)

In [3]:
model.invoke('Hi There!')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--c875b73a-b3f0-4d48-9241-fffdd0d29145-0', usage_metadata={'input_tokens': 4, 'output_tokens': 10, 'total_tokens': 52})

You can set the output format and structure for the model. Langchain support 3 types, Pydantic, TypedDict, JSON

In [6]:
from pydantic import BaseModel, Field

# Pydantic

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")



In [ ]:
# Bind model with structure output format
model_with_structure = model.with_structured_output(Movie)
model2_with_structure = model2.with_structured_output(Movie)
# response = model_with_structure.invoke("Provide details about the movie Inception"
#                                        , include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response2 = model2_with_structure.invoke("Provide details about the movie Inception")

print(response)
# Somehow, Google seem not to work??? 

print(response2)
# OpenAI Work fine


None
title='Inception' year=2010 director='Christopher Nolan' rating=8.8


In [27]:
response2

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

# TypedDict

In [12]:
from typing_extensions import TypedDict, Annotated

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [14]:
model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}

ValueError: no signature found for builtin type <class 'dict'>

In [ ]:
model2_with_structure = model2.with_structured_output(MovieDict)
response2 = model2_with_structure.invoke("Provide details about the movie Inception")
print(response2) 

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


# Json

In [15]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

In [18]:
vars(model_with_structure)

{'name': None,
 'first': RunnableBinding(bound=ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x00000140FA72A350>, async_client=<google.ai.generativelanguage_v1beta.services.generative_service.async_client.GenerativeServiceAsyncClient object at 0x00000140FA75E3D0>, default_metadata=()), kwargs={'tools': [{'function_declarations': [{'name': 'Movie', 'description': 'A movie with details.', 'parameters': {'type_': 6, 'properties': {'rating': {'type_': 2, 'description': "The movie's rating out of 10", 'format_': '', 'nullable': False, 'enum': [], 'properties': {}, 'required': []}, 'title': {'type_': 1, 'description': 'The title of the movie', 'format_': '', 'nullable': False, 'enum': [], 'properties': {}, 'required': []}, 'year': {'type_': 3, 'description': 'The year the movie was released', 'format_': '', 'nullable': False, 'enum

In [16]:
model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)

response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

ValueError: Received unsupported arguments {'method': 'json_schema'}

In [ ]:
model2_with_structure = model2.with_structured_output(
    json_schema,
    method="json_schema",
)

response2 = model2_with_structure.invoke("Provide details about the movie Inception")

None


In [ ]:
print(response2)

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


In [31]:
response2

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}